# 혜진 v10: 유전자 중심 표현·단서 가리기·Bernoulli NB·선형 SVM
새 버전은 표기 상세에 대한 의존을 줄이는 표현과 학습 증강을 비교합니다. 유전자 유무, 유전자+유형, 동의/비동의/구조 유형 채널을 구분하고 학습에서 관측된 변이 유전자 일부만 결측으로 가립니다. 원래 WT를 결측으로 바꾸지 않습니다. 가려도 라벨이 같다는 가정은 검증할 가설이며 실제 결측/측정 과정을 보장하지 않습니다.

노트북 하나에서 데이터·EDA·파싱·선별·학습·검증·피클·CSV·문서를 생성합니다. 이전 피클/프로젝트 Python 파일이 필수가 아니며 캐시가 없으면 v6도 다시 학습합니다. 원본 data/와 requirements-v10.txt 환경은 필요합니다. 반복 사용한 개발 환자/홀드아웃과 기존 seed=109/211은 독립 새 환자가 아닙니다. Public 향상은 아직 알 수 없습니다.

탐색 기록: 초기 19개 결과를 본 뒤, 클래스 균형 sigmoid 보정 SVM 2개와 WT 부재 likelihood를 쓰지 않는 Complement NB 3개를 추가했습니다. 총 24개 설정의 적응 탐색이며 선택 편향이 있습니다. 기본 채택 기준은 바꾸지 않았습니다. Run All은 24개를 처음부터 평가합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v10';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v10 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


In [2]:
# Descriptive EDA only: no test-derived vocabularies or fitted preprocessing.
burden_train=(train[genes].notna()&train[genes].ne('WT')).sum(axis=1)
burden_test=(test[genes].notna()&test[genes].ne('WT')).sum(axis=1)
eda=pd.DataFrame({'train':burden_train.describe(),'test':burden_test.describe()})
eda.to_csv(OUT/'eda_mutated_gene_burden.csv')
byclass=pd.DataFrame({'SUBCLASS':y,'mutated_genes':burden_train}).groupby('SUBCLASS').mutated_genes.agg(['count','mean','median','max'])
byclass.to_csv(OUT/'eda_by_class.csv')
missing=pd.DataFrame({'train_missing':train[genes].isna().sum(),'test_missing':test[genes].isna().sum()})
missing.to_csv(OUT/'eda_missing_by_gene.csv')
print('4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.')
display(eda);display(byclass)


4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.


,train,test
count,6201.000000,2546.000000
mean,35.299629,78.134328
std,100.776379,213.964535
min,0.000000,0.000000
25%,7.000000,11.000000
50%,14.000000,28.000000
75%,27.000000,52.000000
max,2393.000000,2784.000000


,count,mean,median,max
SUBCLASS,,,,
ACC,72,39.430556,24.0,396
BLCA,104,70.490385,54.0,382
BRCA,786,17.437659,9.0,853
CESC,155,44.129032,20.0,480
COAD,223,73.121076,23.0,1631
DLBC,38,67.421053,30.0,704
GBMLGG,461,12.535792,11.0,121
HNSC,223,40.569507,31.0,386
KIPAN,515,16.042718,15.0,246


In [3]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [4]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [5]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [6]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('New configurations',len(NEW))

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Total new configurations including augmentation extension:',len(NEW))


New configurations 8
Total new configurations including augmentation extension: 12


In [7]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


## 1. 유전자 중심 표현과 변이 유전자 가리기
- gene: 유전자 변이 유무 + 관측 변이 유전자 수/결측 수 요약.
- type: 위 정보 + 유전자별 동의/치환/stop/fs/del/ins/delins/미해석 유형 존재.
- functional: 유전자 유무 + 동의·비동의·구조 유형·미해석 채널, 해당 유전자 수/비율 요약. 구조 유형 표기는 기능 상실 확정이 아닙니다.
- 범위/서열 길이가 맞는 복합 치환만 개별 치환으로 분해합니다. 불완전 표기는 추측하지 않습니다.
- mask: 해당 학습 환자의 관측 변이 유전자를 확률 0.1/0.25로 결측 처리한 복사본. 원본:복사본 비중 1:0.5를 유지하되 총 sample weight가 원본 환자 수와 같게 정규화해 규제 강도 변화와 구분합니다. 사전/최소 빈도는 원본 학습만 사용합니다.
- NB에는 이진 변수만 넣고, 결측은 관측 WT로 계산하지 않는 Bernoulli likelihood를 사용합니다. 실제 결측이 생물학적으로 WT라는 뜻이 아닙니다.

In [8]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_selection import SelectKBest

def expand10(raw):
 t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)
 if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):
  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))
 return (t,)

def kind10(t):
 if 'fs' in t:return 'frameshift'
 if t.endswith('*'):return 'stop'
 if 'delins' in t:return 'delins'
 if 'del' in t:return 'deletion'
 if 'ins' in t:return 'insertion'
 m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)
 if m:return 'synonymous' if m[1]==m[2] else 'substitution'
 return 'other'

def packet10(frame,ev=None):
 if ev is None:ev=parse_events(frame[genes])
 resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]
 missing=frame[genes].isna().to_numpy()
 return dict(events=resolved,missing=missing)

def records10(packet,representation,numeric=True):
 rs=[]
 for i,row in enumerate(packet['events']):
  d={};syn=nonsyn=structural=unparsed=0
  for g,ts in row:
   d['gene:'+g]=1.;types={kind10(t) for t in ts}
   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types
   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot
   if representation=='type':
    for typ in types:d['type:'+g+':'+typ]=1.
   elif representation=='functional':
    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:
     if flag:d[family+':'+g]=1.
  if numeric:
   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))
   if representation!='gene':
    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:
     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)
  rs.append(d)
 return rs

def mask_packet10(packet,probability,seed):
 rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}
 for i,row in enumerate(packet['events']):
  keep=[]
  for g,ts in row:
   if rng.random()<probability:missing[i,gpos[g]]=True
   else:keep.append((g,ts))
  out.append(keep)
 return dict(events=out,missing=missing)

def select_cols10(a,target,k):
 if k is None or a.shape[1]<=k:return np.arange(a.shape[1])
 with warnings.catch_warnings():
  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
 return np.sort(np.argsort(-scores,kind='stable')[:k])

class MissingBernoulli10:
 def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k
 def fit(self,a,target,missing,sample_weight=None):
  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])
  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  observed=nclass[:,None]-absent
  if self.pooled:
   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()
   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)
   prob=(counts+self.alpha*prior)/(observed+self.alpha)
  else:prob=(counts+self.alpha)/(observed+2*self.alpha)
  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0
  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)
  self.coef_=self.logit;return self
 def predict_proba(self,a,missing):
  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])
  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)
  return softmax(score,axis=1)

class GroupSigmoidSVM10:
 def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration
 def fit(self,a,target,profile_groups,sample_weight):
  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)
  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):
   assert not set(profile_groups[tr]).intersection(profile_groups[va])
   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])
   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])
   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))
   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores
  self.calibrators=[]
  for j,c in enumerate(self.classes_):
   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)
  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self
 def predict_proba(self,a):
  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))
  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])
  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)

class Member10:
 def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):
  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler
  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])
 def missing_matrix(self,packet):
  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0
  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]
  return csr32(sparse.csr_matrix(out))
 def predict_packet(self,packet,records=None):
  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))
  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))
  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)
  tx=a[:,self.cols]
  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))
  return self.est.predict_proba(tx)
 def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 packet=None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   if packet is None:packet=packet10(frame[genes])
   q=m.predict_packet(packet)*w;p=q if p is None else p+q
 return p

C10={
 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),
 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),
 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),
 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),
 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),
 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),
 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),
 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),
 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),
 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),
 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),
 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),
 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),
 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),
 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}

def prepare10(idx,representation,numeric,cache):
 key=(representation,numeric)
 if key not in cache:
  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])
  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)
 return cache[key]

def train10(name,idx,cache):
 cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]
 augmented=None
 if cfg['mask']:
  key=('masked',cfg['mask'])
  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))
  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5
 if cfg['algo']=='nb':
  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])
  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False)
 if cfg['algo']=='cnb':
  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False,cols=cols)
 if cfg['algo']=='svm':
  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)
 # Outer selection and scaler are fit on the original training patients only.
 cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])
 scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None
 tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)
 est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)
 est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)

# A missing feature must contribute no Bernoulli presence or absence evidence.
toy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)
nbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])
assert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')

C10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))
print('Extended candidates:',len(C10))


V10 candidates: 19 semantic checks passed


Five calibrated/Complement NB settings added; 24 total.


## 2. v6 고정 대조군의 완료 OOF 캐시
v9에서 저장한 구성별 OOF와 데이터 해시/ID/프로필/클래스/분할을 확인합니다. 원래 v6 비율로 재구성한 OOF가 v6 완료 결과와 일치해야 합니다. 반복 분할 seed=109/211의 v6 구성별 확률도 같은 데이터/분할의 완료 결과만 재사용합니다. 새 v10 모델은 해당 fold에서 실제로 학습하며 전체 train 캐시는 최종 전체 학습에서만 허용합니다. 캐시가 없거나 CACHE_REFERENCE=False면 v6를 다시 학습합니다.

In [9]:
CACHE_REFERENCE=True
reference_cache10=None;repeat_cache10={};v9_dir=WORK/'artifacts'/'v9';v6_dir=WORK/'artifacts'/'v6';cache_valid10=False
W6_10=np.array([.0625,.0625,.0625,.140625,.140625,.140625,.140625,.25])
if CACHE_REFERENCE and all(p.exists() for p in [v9_dir/'verification.json',v9_dir/'component_oof.npz',v9_dir/'split_manifest.csv']):
 proof9=json.loads((v9_dir/'verification.json').read_text())
 for name in ['train.csv','test.csv','sample_submission.csv']:assert hashlib.sha256((DATA/name).read_bytes()).hexdigest()==proof9['data_sha256'][name]
 manifest=pd.read_csv(v9_dir/'split_manifest.csv',dtype={'group':str});assert manifest.ID.equals(train.ID) and np.array_equal(manifest.group.to_numpy(),groups.astype(str))
 q=np.load(v9_dir/'component_oof.npz',allow_pickle=True);assert np.array_equal(q['dev'],dev) and np.array_equal(q['labels'],labels) and np.array_equal(np.array(proof9['component_names']),q['names'])
 reference_cache10=np.tensordot(W6_10,q['probabilities'],axes=(0,0))
 q6=np.load(v6_dir/'oof_probabilities.npz',allow_pickle=True);assert np.allclose(reference_cache10,.75*q6['v5_reference']+.25*q6['gated_s3_aug_global_lgb4096'],atol=1e-6)
 if (v9_dir/'diagnostic_component_probabilities.npz').exists():
  qr=np.load(v9_dir/'diagnostic_component_probabilities.npz',allow_pickle=True);assert np.array_equal(qr['dev'],dev) and np.array_equal(qr['labels'],labels) and np.array_equal(qr['names'],q['names'])
  for seed in [109,211]:repeat_cache10[seed]=np.tensordot(W6_10,qr[f'seed_{seed}'],axes=(0,0))
 cache_valid10=True
print('Reference primary cache:',cache_valid10,'repeat seeds:',list(repeat_cache10))


Reference primary cache: True repeat seeds: [109, 211]


## 3. 전처리·분류기 비교
새 단일 모델과 v6에 25/50/75% 혼합하는 계획을 같은 그룹 CV에서 비교합니다. 원본 F1 중심 후보(+0.003·스트레스 하락 0.005 이하·종합 비하락)와 종합 중심 후보(+0.002·원본 하락 0.002 이하)를 최대 2개 고정합니다. 이후 동일 seed=109/211에서 새 후보를 재학습하고 이전 v6 대조군과 비교합니다. 이 seed들도 이전에 관측한 개발 환자 재분할이라 독립 새 환자는 아닙니다.

In [10]:
P10={name:np.zeros((3,len(dev),NC),np.float32) for name in ['v6_reference',*C10]};rows10=[]
for fold,(a,b) in enumerate(folds):
 tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,SEED+901+fold);packets=[packet10(f,ev) for f,ev in zip(frames,evs)]
 if cache_valid10:P10['v6_reference'][:,b]=reference_cache10[:,b]
 else:
  bm,bw=fit_v6(tr)
  for j,f in enumerate(frames):P10['v6_reference'][j,b]=predict_members(bm,bw,f)
  del bm
 cache={};record_cache={}
 for name,cfg in C10.items():
  start=time.time();m=train10(name,tr,cache)
  for j,packet in enumerate(packets):
   key=(cfg['rep'],cfg['algo'] not in ['nb','cnb'],j)
   if key not in record_cache:record_cache[key]=records10(packet,cfg['rep'],cfg['algo'] not in ['nb','cnb'])
   P10[name][j,b]=m.predict_packet(packet,record_cache[key])
  assert np.allclose(P10[name][0,b],P10[name][2,b]),'Duplicate annotation should not change new representations'
  print('V10',fold,name,'inputs',len(m.names),'seconds',round(time.time()-start,1),flush=True);del m
 for name,p in P10.items():
  fs=[f1_score(yi[va],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
  rows10.append(dict(model=name,fold=fold,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 pd.DataFrame(rows10).to_csv(OUT/'cv_folds.csv',index=False);np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,**P10)
 print('Completed primary fold',fold,flush=True)

def score_plan10(p,splits):
 fs=np.array([[f1_score(yi[dev[b]],p[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)] for _,b in splits])
 return dict(clean=fs[:,0].mean(),noise=fs[:,1].mean(),duplicate=fs[:,2].mean(),score=(fs@np.array([.6,.3,.1])).mean())
PLANS10={'v6_reference':[('v6_reference',1.)]}
for name in C10:
 PLANS10[name]=[(name,1.)]
 for w in [.25,.5,.75]:PLANS10[f'v6plus_{name}_{w}']=[('v6_reference',1-w),(name,w)]
rank10=pd.DataFrame({name:score_plan10(sum(P10[k]*w for k,w in plan),folds) for name,plan in PLANS10.items()}).T.sort_values('score',ascending=False);rank10.to_csv(OUT/'model_comparison.csv')
base=rank10.loc['v6_reference'];finalists10=[];objectives10={}
clean=rank10[(rank10.index!='v6_reference')&(rank10.clean>=base.clean+.003)&(rank10.noise>=base.noise-.005)&(rank10.score>=base.score)]
if len(clean):
 name=clean.sort_values('clean',ascending=False).index[0];finalists10.append(name);objectives10[name]='clean'
robust=rank10[(rank10.index!='v6_reference')&(rank10.score>=base.score+.002)&(rank10.clean>=base.clean-.002)]
if len(robust):
 name=robust.index[0]
 if name not in finalists10:finalists10.append(name);objectives10[name]='robust'
(OUT/'selection_primary.json').write_text(json.dumps(dict(finalists=finalists10,objectives=objectives10,plans={name:PLANS10[name] for name in finalists10}),indent=2))
print('Frozen finalists:',finalists10,objectives10,flush=True);display(rank10.head(15))


Scoring 19 completed settings plus 5 additional calibrated/Complement NB settings.
Frozen finalists: ['v6plus_type_complement_nb_a0.1_0.25'] {'v6plus_type_complement_nb_a0.1_0.25': 'clean'}


,clean,noise,duplicate,score
v6plus_type_complement_nb_a0.1_0.25,0.442404,0.345993,0.440209,0.413261
v6plus_functional_complement_nb_a1_0.25,0.440854,0.348787,0.439661,0.413115
v6plus_type_complement_nb_a1_0.25,0.441618,0.346202,0.440754,0.412906
v6plus_functional_svm_balanced_mask10_0.25,0.437104,0.347210,0.432816,0.409707
v6plus_functional_svm_balanced_0.5,0.434833,0.347976,0.435074,0.408800
v6plus_functional_lr_mask10_0.25,0.433009,0.352193,0.432976,0.408761
v6plus_functional_svm_balanced_0.25,0.435175,0.347951,0.432517,0.408742
v6plus_functional_svm_balanced_mask10_0.5,0.434445,0.348113,0.434013,0.408502
v6_reference,0.435576,0.345775,0.433301,0.408408
v6plus_functional_lr_0.25,0.432373,0.352125,0.432392,0.408301


## 4. 고정 후보 반복 검증
원본 중심 후보는 두 seed 평균 clean +0.002 이상·종합 비하락, 종합 후보는 clean +0.001 이상·종합 +0.001 이상을 요구합니다. 모든 seed에서 clean 하락 0.002 이하·스트레스 하락 0.005 이하도 요구합니다. 후보/혼합 비율은 확인 결과를 보고 다시 조정하지 않습니다. 캐시가 있으면 기존 v6 대조 확률만 재사용하고 새 전처리·SVM 보정·NB 통계는 각 외부 학습에서 다시 적합합니다.

In [16]:
from concurrent.futures import ThreadPoolExecutor,as_completed
confirm_rows10=[];confirm_probs10={};CONFIRM_SEEDS10=[109,211]
needed10=list(dict.fromkeys(k for name in finalists10 for k,w in PLANS10[name] if k!='v6_reference'))
for seed in CONFIRM_SEEDS10:confirm_probs10[seed]={name:np.zeros((3,len(dev),NC),np.float32) for name in ['v6_reference',*needed10]}

def confirm_task10(seed,fold,a,b):
 tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,SEED+seed*10+fold);packets=[packet10(f,ev) for f,ev in zip(frames,evs)];local={};result={}
 if seed in repeat_cache10:result['v6_reference']=repeat_cache10[seed][:,b]
 else:
  bm,bw=fit_v6(tr);result['v6_reference']=np.stack([predict_members(bm,bw,f) for f in frames]);del bm
 for name in needed10:
  m=train10(name,tr,local);result[name]=np.stack([m.predict_packet(packet) for packet in packets]);del m
 rows=[]
 for name in ['v6_reference',*finalists10]:
  p=sum(result[k]*w for k,w in PLANS10[name]);fs=[f1_score(yi[va],p[j].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
  rows.append(dict(seed=seed,fold=fold,model=name,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 return seed,fold,b,result,rows

if finalists10:
 tasks=[]
 for seed in CONFIRM_SEEDS10:
  for fold,(a,b) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(X.iloc[dev],y[dev],groups[dev])):
   assert not set(groups[dev[a]]).intersection(groups[dev[b]]);tasks.append((seed,fold,a,b))
 with ThreadPoolExecutor(max_workers=2) as pool:
  for future in as_completed([pool.submit(confirm_task10,*task) for task in tasks]):
   seed,fold,b,ps,rows=future.result()
   for name,prob in ps.items():confirm_probs10[seed][name][:,b]=prob
   confirm_rows10.extend(rows);pd.DataFrame(confirm_rows10).to_csv(OUT/'confirmation_cv.csv',index=False)
   np.savez_compressed(OUT/'confirmation_probabilities.npz',dev=dev,labels=labels,**{f's{seed}__{name}':p for seed,ps in confirm_probs10.items() for name,p in ps.items()})
   print('Confirmation',seed,fold,rows,flush=True)
 confirmation10=pd.DataFrame(confirm_rows10).groupby(['seed','model'])[['clean','noise','duplicate','score']].mean()
else:confirmation10=pd.DataFrame();print('No candidate passed primary gates.')
passes10={};deltas10={}
for name in finalists10:
 delta=pd.DataFrame([confirmation10.loc[(seed,name)]-confirmation10.loc[(seed,'v6_reference')] for seed in CONFIRM_SEEDS10],index=CONFIRM_SEEDS10)
 bounded=bool((delta.clean>=-.002).all() and (delta.noise>=-.005).all());mean=delta.mean()
 gain=bool(mean.clean>=.002 and mean.score>=0) if objectives10[name]=='clean' else bool(mean.clean>=.001 and mean.score>=.001)
 passes10[name]=bounded and gain;deltas10[name]=dict(per_seed=delta.to_dict(orient='index'),mean=mean.to_dict())
winner10=next((name for name in finalists10 if passes10[name]),'v6_reference');accepted10=winner10!='v6_reference';final_plan10=PLANS10[winner10]
(OUT/'selection.json').write_text(json.dumps(dict(winner=winner10,accepted=accepted10,plan=final_plan10,passes=passes10,deltas=deltas10),indent=2))
print('Final winner:',winner10,'adopted:',accepted10,flush=True);display(confirmation10)


Confirmation 109 0 [{'seed': 109, 'fold': 0, 'model': 'v6_reference', 'clean': 0.4290683040442191, 'noise': 0.30905629028498743, 'duplicate': 0.42991080757616407, 'score': 0.39314895026964414}, {'seed': 109, 'fold': 0, 'model': 'v6plus_type_complement_nb_a0.1_0.25', 'clean': 0.43353049066247007, 'noise': 0.3049515315629796, 'duplicate': 0.433511196378721, 'score': 0.39495487350424796}]
Confirmation 109 1 [{'seed': 109, 'fold': 1, 'model': 'v6_reference', 'clean': 0.438201965060693, 'noise': 0.340747012003311, 'duplicate': 0.4353681998145831, 'score': 0.4086821026188674}, {'seed': 109, 'fold': 1, 'model': 'v6plus_type_complement_nb_a0.1_0.25', 'clean': 0.4367355161283918, 'noise': 0.34125054222806883, 'duplicate': 0.4433555016075494, 'score': 0.4087520225062107}]
Confirmation 109 2 [{'seed': 109, 'fold': 2, 'model': 'v6_reference', 'clean': 0.4270400499382076, 'noise': 0.3325822367266885, 'duplicate': 0.42754906028147677, 'score': 0.3987536070090788}, {'seed': 109, 'fold': 2, 'model': '

clean  ...     score
seed model                                          ...          
109  v6_reference                         0.431437  ...  0.400195
     v6plus_type_complement_nb_a0.1_0.25  0.436537  ...  0.403848
211  v6_reference                         0.435268  ...  0.407771
     v6plus_type_complement_nb_a0.1_0.25  0.436980  ...  0.406851

[4 rows x 4 columns]

## 5. 이전 홀드아웃·전체 학습·피클·CSV
선택은 이미 고정했습니다. 홀드아웃은 반복 관측한 참고값이며 이를 보고 가중치를 다시 고르지 않습니다. 전체 train 캐시는 최종 전체 학습에서만 사용하고 외부 검증/홀드아웃에는 사용하지 않습니다. 미채택이면 주 모델 v6를 유지하고 동일 CSV를 새 제출로 배포하지 않습니다.

In [18]:
def fit_plan10(idx,plan,full=False):
 cache={};ms=[];ws=[]
 for name,w in plan:
  if name=='v6_reference':
   if full and cache_valid10 and (v6_dir/'cancer_model_v6.pkl').exists():
    with open(v6_dir/'cancer_model_v6.pkl','rb') as f:old=pickle.load(f)
    assert old['version']==6 and old['genes']==genes and np.array_equal(old['classes'],labels);bm,bw=old['models'],np.array(old['weights'])
   else:bm,bw=fit_v6(idx)
   ms.extend(bm);ws.extend(bw*w)
  else:ms.append(train10(name,idx,cache));ws.append(w)
 return ms,np.array(ws,float)

hm,hw=fit_plan10(dev,final_plan10);hp=predict_members(hm,hw,X.iloc[hold]);hpred=labels[hp.argmax(axis=1)]
metrics10=dict(macro_f1=f1_score(y[hold],hpred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],hpred),weighted_f1=f1_score(y[hold],hpred,average='weighted',zero_division=0),n=len(hold))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics10,indent=2));np.savez_compressed(OUT/'holdout_probabilities.npz',probabilities=hp,indices=hold)
pd.DataFrame(classification_report(y[hold],hpred,output_dict=True,zero_division=0)).T.to_csv(OUT/'holdout_classification_report.csv')
print('Previously observed holdout:',metrics10,flush=True);del hm
models10,weights10=fit_plan10(np.arange(len(train)),final_plan10,full=True)
rows=[];sizes10=[]
for m,w in zip(models10,weights10):
 sizes10.append(dict(model=m.name,view=m.view,inputs=len(m.names),weight=float(w),representation=getattr(m,'representation',None),mask_rate=C10[m.name]['mask'] if m.name in C10 else None))
 for feature in m.raw_names:rows.append(dict(model=m.name,view=m.view,feature=feature))
pd.DataFrame(rows).to_csv(OUT/'final_features.csv',index=False);(OUT/'model_sizes.json').write_text(json.dumps(sizes10,indent=2))
def predict_bundle10(bundle,frame):return predict_members(bundle['models'],bundle['weights'],frame)
bundle10=dict(version=10,models=models10,weights=weights10,genes=genes,classes=labels,winner=winner10,holdout_metrics=metrics10,plan=final_plan10,predict=predict_bundle10)
model_path10=OUT/'cancer_model_v10.pkl'
with model_path10.open('wb') as f:cloudpickle.dump(bundle10,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path10.open('rb') as f:loaded10=pickle.load(f)
tp10=loaded10['predict'](loaded10,test);assert np.isfinite(tp10).all() and np.allclose(tp10.sum(axis=1),1)
submission10=sample.copy();submission10.SUBCLASS=labels[tp10.argmax(axis=1)];assert submission10.ID.equals(test.ID);submission10.to_csv(OUT/'submission_v10.csv',index=False)
if accepted10:submission10.to_csv(SUBMIT/'혜진_submission_v10.csv',index=False)
pd.DataFrame(tp10,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('MODEL',model_path10,'MiB',round(model_path10.stat().st_size/1024**2,2),'new submission exported:',accepted10,flush=True)


Previously observed holdout: {'macro_f1': 0.4729888723709301, 'accuracy': 0.46333601933924257, 'weighted_f1': 0.46777775879200023, 'n': 1241}
MODEL /Users/mac/project/cancer_project/artifacts/v10/cancer_model_v10.pkl MiB 83.09 new submission exported: False


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


## 6. 독립 재로딩·문서·변수 사전

In [20]:
import subprocess,tempfile,importlib.metadata
program="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);s=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert s.ID.equals(x.ID) and np.array_equal(b['classes'][p.argmax(axis=1)],s.SUBCLASS.to_numpy())
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 pf=Path(td)/'p.npy';rr=subprocess.run([sys.executable,'-c',program,str(model_path10.resolve()),str((DATA/'test.csv').resolve()),str((OUT/'submission_v10.csv').resolve()),str(pf)],cwd=td,env=env,capture_output=True,text=True)
 if rr.returncode:raise RuntimeError(rr.stderr)
 assert np.allclose(np.load(pf),tp10)
proof10=dict(status='passed',version=10,accepted_new_candidate=accepted10,winner=winner10,plan=final_plan10,finalists=finalists10,objectives=objectives10,passes=passes10,deltas=deltas10,primary=rank10.loc[winner10].to_dict(),baseline=rank10.loc['v6_reference'].to_dict(),confirmation=confirmation10.reset_index().to_dict(orient='records') if len(confirmation10) else [],holdout=metrics10,configurations=len(C10),model_bytes=model_path10.stat().st_size,reference_cache_validated=cache_valid10,reference_repeat_seeds=list(repeat_cache10),full_independent_test_reload='passed',csv_exact_match=True,batch_invariance=True,public_score=None,python=sys.version,data_sha256={f:hashlib.sha256((DATA/f).read_bytes()).hexdigest() for f in ['train.csv','test.csv','sample_submission.csv']},packages={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','cloudpickle','matplotlib','nbformat','ipython','threadpoolctl']})
(OUT/'verification.json').write_text(json.dumps(proof10,indent=2));(WORK/'requirements-v10.txt').write_text('# Python '+platform.python_version()+'\n'+'\n'.join(k+'=='+v for k,v in proof10['packages'].items())+'\n')

def mdtable10(frame):
 lines=['| '+' | '.join(map(str,frame.columns))+' |','| '+' | '.join(['---']*len(frame.columns))+' |']
 for row in frame.itertuples(index=False,name=None):lines.append('| '+' | '.join(f'{v:.6f}' if isinstance(v,(float,np.floating)) else str(v) for v in row)+' |')
 return '\n'.join(lines)
importance10=[]
for m,w in zip(models10,weights10):
 if hasattr(m.est,'feature_importances_'):values=m.est.feature_importances_;kind='tree_importance'
 elif isinstance(m.est,GroupSigmoidSVM10):values=np.max(np.abs(m.est.coef_),axis=0);kind='max_abs_precalibration_svm_coefficient'
 elif isinstance(m.est,MissingBernoulli10):values=np.max(np.abs(m.est.coef_),axis=0);kind='max_abs_bernoulli_log_odds'
 elif hasattr(m.est,'feature_log_prob_'):values=np.max(np.abs(m.est.feature_log_prob_),axis=0);kind='max_abs_complement_log_weight'
 elif hasattr(m.est,'coef_'):values=np.max(np.abs(m.est.coef_),axis=0);kind='max_absolute_coefficient'
 else:values=np.linalg.norm(m.est.coefs_[0],axis=1);kind='first_layer_weight_norm'
 assert len(values)==len(m.names)
 for feature,value in zip(m.names,values):importance10.append(dict(model=m.name,view=m.view,feature=feature,importance=float(value),metric=kind,ensemble_weight=float(w)))
importance10=pd.DataFrame(importance10);importance10.to_csv(OUT/'final_feature_importance.csv',index=False)
meaning10={'gene':'유전자 변이 관측 여부 또는 기존 가중 존재','type':'유전자에서 관측된 변이 유형 존재','syn_gene':'동의 변이 표기가 있는 유전자','nonsyn_gene':'치환·stop/fs/del/ins/delins 유형이 있는 유전자','structural_gene':'stop/fs/del/ins/delins 표기; 기능 상실 확정 아님','unparsed_gene':'문법상 미해석 변이가 있는 유전자','numeric':'관측 변이 수·결측 수 로그, 유형별 유전자 수/비율 등 요약','variant':'정규화한 정확한 단백질 변이','position':'유전자와 단백질 위치','aa':'유전자와 아미노산 변화 종류','supported':'학습에서 지지된 비동의 변이 존재','module':'명시한 유전자 목록의 가중 변이 집계','missing':'결측 표기'}
importance10['meaning']=importance10.feature.map(lambda x:meaning10.get(x.split(':')[0],'노트북 정의 참조'));importance10.to_csv(OUT/'final_feature_dictionary.csv',index=False)
(WORK/'selected_variables_v10.md').write_text('# v10 최종 유지/채택 모델의 실제 변수\n\n학습 내 빈도·선별과 개발/재분할 기준으로 선택한 실제 모델 입력입니다. 미채택이면 유지한 v6의 변수입니다. 중요도는 서로 다른 모델 사이에서 비교하거나 인과효과로 해석하지 않습니다. SVM 값은 확률 보정 전 마진의 계수이고 NB 값은 관측 Bernoulli log-odds입니다.\n\n'+mdtable10(importance10)+'\n')
families10={'gene','type','syn_gene','nonsyn_gene','structural_gene','unparsed_gene','variant','position','aa','supported','missing'}
links10=importance10.copy();links10['gene']=links10.feature.map(lambda f:f.split(':')[1] if f.split(':')[0] in families10 else '')
counts10=links10[links10.gene.ne('')].groupby('gene').size();views10=links10[links10.gene.ne('')].groupby('gene').view.agg(lambda x:', '.join(sorted(set(x))))
mu=(train[genes].notna()&train[genes].ne('WT'));mt=(test[genes].notna()&test[genes].ne('WT'))
gdict10=pd.DataFrame(dict(gene=genes,train_mutated=mu.sum().values,train_rate=mu.mean().values,test_rate=mt.mean().values));gdict10['selected_model_feature_uses']=gdict10.gene.map(counts10).fillna(0).astype(int);gdict10['selected_views']=gdict10.gene.map(views10).fillna('-');gdict10.to_csv(OUT/'gene_dictionary_v10.csv',index=False)
(WORK/'gene_dictionary_v10.md').write_text('# v10 전체 유전자 컬럼 사전\n\n관측 통계이며 기능/병원성 주석은 추정하지 않습니다. test_rate는 기술통계만 사용했습니다. 연결 변수 수는 모델 간 중복을 포함하고 0이어도 숫자/경로 요약에 기여할 수 있습니다.\n\n'+mdtable10(gdict10)+'\n')
# Paired masking comparisons use otherwise identical model settings.
pairs10=[('gene_lr','gene_lr_mask10'),('gene_lr','gene_lr_mask25'),('type_lr','type_lr_mask10'),('functional_lr','functional_lr_mask10'),('functional_svm','functional_svm_mask10'),('functional_lgb','functional_lgb_mask10'),('functional_lgb','functional_lgb_mask25')]
masking_rows10=[]
for old,new in pairs10:
 delta=rank10.loc[new]-rank10.loc[old];masking_rows10.append(dict(reference=old,masked=new,delta_clean=delta.clean,delta_noise=delta.noise,delta_duplicate=delta.duplicate,delta_score=delta.score))
pd.DataFrame(masking_rows10).to_csv(OUT/'masking_ablation.csv',index=False)
caption='채택한 v10' if accepted10 else '유지 모델 v6'
report10=f'''# 혜진 v10: 유전자 중심 표현·단서 가리기·새 분류기 비교

## 결과
초기 19개에 보정/likelihood 대조 5개를 추가한 24개 단일 설정과 고정 v6 혼합을 동일 그룹 CV에서 비교했습니다. 원본 F1 중심/종합 점수 중심 후보를 최대 2개 고정하고 같은 개발 환자의 seed=109/211에서 새 후보를 다시 학습했습니다. 최종 `{winner10}`, 신규 채택 **{accepted10}**입니다.

{caption}의 기존 홀드아웃 Macro F1 {metrics10['macro_f1']:.6f}, 정확도 {metrics10['accuracy']:.6f}, weighted F1 {metrics10['weighted_f1']:.6f}. 미채택이면 이 점수는 v6 재현값이고 신규 후보 성능이 아닙니다. 개발/홀드아웃/동일 seed는 이전 버전에서도 관측했으므로 새 독립 성능이 아닙니다. Public은 미확인입니다.

## 표현과 파싱
정확한 변이·위치·AA 변화 등 많은 단서를 한꺼번에 사용하는 표현과 달리, gene은 유전자 유무, type은 유전자와 유형 존재, functional은 유전자 유무와 동의·비동의·구조·미해석 채널을 사용합니다. NB를 제외한 모델은 관측 변이 유전자 수와 결측 수를 log1p로 보존합니다. type/functional은 유형별 유전자 수/비율 요약도 유지합니다. row 전체 L2 정규화는 하지 않았습니다.

범위와 서열 길이가 맞는 복합 치환만 분해하고, uppercase 아미노산 FS를 lowercase fs 접미사와 구분했습니다. 불완전 변이는 미해석으로 남기고 WT와 실제 결측은 구분했습니다. 변이 수는 mutations/Mb TMB가 아니며 구조 유형이 기능 상실을 뜻하지 않습니다.

## 가리기 증강
학습의 관측 변이 유전자만 10% 또는 25% 확률로 가린 복사본을 추가합니다. 가린 칸은 결측으로 처리하고 실제 WT를 결측으로 바꾸지 않습니다. 원본:복사본 sample weight 비율은 1:0.5이고 총 weight가 원본 환자 수와 같게 정규화해 규제 손잡이 변화와 구분했습니다. 사전/min_df=3은 원본 학습만 사용합니다.

이 방식은 관측 단서 일부를 잃어도 라벨이 같다는 가정입니다. 실제 결측 생성 과정이나 유전자의 기능을 재현하지 않습니다. 변이만 선택해 가리는 것이 실제 결측과 다를 수 있고, 학습 결측이 원래 0인 데이터에서 만든 합성 조건이라는 한계가 있습니다.

## Bernoulli Naive Bayes
이진 유전자/유형 존재만 사용하며 숫자 요약은 넣지 않습니다. 관측이 없는 특징은 존재·부재 likelihood 모두 계산에서 제외해 관측 WT와 구분합니다. 보통의 Beta(alpha,alpha) smoothing alpha=0.1/1, 경험 class prior 비교와 전체 학습 유전자 빈도를 prior 평균으로 삼는 pooled strength=20 smoothing을 비교했습니다. 모두 학습 부분으로만 적합합니다.

초기 19개 결과를 보고 클래스 균형 보정 SVM 2개와 Complement NB 3개를 추가했습니다. 이는 적응 탐색으로 선택 편향이 있습니다. Complement NB는 관측된 양성 변이 정보의 보완 클래스 빈도를 사용하며, 관측 WT 부재 likelihood는 합산하지 않습니다. alpha=0.1/1, type/functional 표현을 비교했습니다.

유전자 간 조건부 독립 가정은 실제로 성립하지 않을 수 있고, gene/type 채널의 상관은 확률을 과신하게 할 수 있습니다. equal class prior는 Macro F1 관점의 대조 가설이지 test 암종 비율을 강제한 것이 아닙니다. 입력 수는 선택한 이진 변이 변수 수이며 같은 크기의 관측 여부 마스크를 보조 입력으로 사용합니다.

## 선형 SVM과 보정
LinearSVC C=0.1, 클래스 균형, 최대 5,000회와 내부 그룹 3-fold를 사용합니다. 각 내부 학습에서 chi2 선별과 MaxAbs scaler를 적합한 후 외부 내부-fold 마진을 생성합니다. 동일 원본 프로필과 가림 복사본은 같은 내부 그룹을 유지합니다. 이 OOF 마진에 암종별 sigmoid 로지스틱을 적합합니다. 기본은 학습 class prior를 따르고, 추가 대조는 sigmoid의 binary class weight를 balanced로 맞춰 Macro F1에 맞는 순위/비중 효과를 비교합니다. 균형 보정 출력은 실제 모집단의 임상 확률이라는 뜻이 아닙니다. 최종 SVM은 외부 학습 전체로 재학습합니다.

이후 최종 SVM은 외부 학습 전체로 재학습합니다. 예측에서는 각 sigmoid 출력을 합이 1이 되게 정규화합니다. 보정 전 마진을 확률인 것처럼 바로 평균하지 않습니다.

사전/min_df는 외부 학습의 원본 환자로 적합한 공통 vocabulary이며 내부 보정 fold의 라벨을 쓰지 않습니다. 내부 타깃 선별/스케일러/SVM은 그 내부 학습만 사용합니다. 외부 검증/test 라벨이나 빈도는 보정에 쓰지 않습니다. LR은 C=0.1, LGB는 트리 300개·학습률 0.04·잎 15·L2 규제 10으로 비교했습니다.

## 단일 설정 결과
{mdtable10(rank10.loc[list(C10)].reset_index(names='candidate'))}

## 가리기 효과: 같은 학습기와 표현끼리 비교
{mdtable10(pd.DataFrame(masking_rows10))}

## 앙상블·반복 확인
새 단일 모델과 v6에 25/50/75%를 섞는 계획을 비교했습니다. 원본 중심은 개발 clean +0.003, 스트레스 하락 0.005 이하, 종합 비하락을 요구하고, 종합 중심은 종합 +0.002, clean 하락 0.002 이하를 요구합니다. 종합은 기존 0.6 clean/0.3 희귀 변이/0.1 중복입니다. 후보·혼합 비율은 반복 확인 전 고정했습니다.

반복 확인은 모든 seed에서 clean 하락 0.002 이하·스트레스 하락 0.005 이하를 요구합니다. 두 seed 평균에서 clean 후보는 원본 +0.002 이상·종합 비하락, robust 후보는 원본 +0.001 이상·종합 +0.001 이상이어야 합니다. 기준을 후속 점수가 작다는 이유로 낮추지 않습니다. 반복 분할도 같은 환자/이미 사용한 seed라 독립 최종 검증이 아닙니다. 선택 편향과 합성 스트레스의 현실성 한계가 남습니다.

고정 후보 `{finalists10}`, 통과 `{passes10}`.

{mdtable10(confirmation10.reset_index()) if len(confirmation10) else '1차 기준 통과 후보가 없어 반복 확인을 실행하지 않았습니다.'}

## 전체 비교
{mdtable10(rank10.reset_index(names='plan'))}

## 재현과 파일
v6 캐시는 v9에서 저장한 데이터 해시·ID/프로필·클래스·분할을 확인하고, 원래 가중치로 재구성한 값이 v6 OOF와 같을 때만 재사용합니다. 반복 seed도 이전에 그 fold 학습만으로 만든 v6 OOF 대조값을 재사용하며 새 후보는 각 학습에서 실제로 적합합니다. 전체 train 모델은 CV/홀드아웃에 쓰지 않고 최종 전체 학습에서만 재사용합니다. 캐시가 없거나 CACHE_REFERENCE=False면 모두 다시 학습합니다.

cencer_v10.ipynb 하나에 모든 학습·선별·SVM 보정·확인·저장을 포함했습니다. 프로젝트 Python 파일이나 이전 피클은 필수가 아닙니다. 원본 data/와 requirements-v10.txt 환경이 필요합니다. 실제 변수 selected_variables_v10.md, 전체 유전자 통계 gene_dictionary_v10.md, 모델별 정의 model_sizes.json, 가리기 대조 masking_ablation.csv에 기록했습니다. 중요도는 서로 다른 모델 사이에서 비교하거나 인과효과로 해석하지 않습니다.

모델 artifacts/v10/cancer_model_v10.pkl 크기 {model_path10.stat().st_size/1024**2:.2f} MiB입니다. 프로젝트 코드 없는 임시 폴더에서 전체 test 2,546개의 CSV·ID·확률·배치 일치를 확인했습니다. 채택 시 submission/혜진_submission_v10.csv를 배포하고, 미채택 시 유지 모델 v6의 동일 재현 CSV를 새 제출로 배포하지 않습니다. artifacts는 기존 Git 제외 방침을 유지합니다.
'''
(WORK/'analysis_report_v10.md').write_text(report10)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
show=rank10.head(7)
if 'v6_reference' not in show.index:show=pd.concat([show,rank10.loc[['v6_reference']]])
fig,ax=plt.subplots(figsize=(12,6));xx=np.arange(len(show));ax.barh(xx-.18,show.clean,height=.35,label='Clean macro F1');ax.barh(xx+.18,show.noise,height=.35,label='Rare-variant stress macro F1');ax.set_yticks(xx,show.index);ax.invert_yaxis();ax.legend();fig.tight_layout();fig.savefig(OUT/'validation_comparison_v10.png',dpi=150);plt.close(fig)
p=WORK/'analysis_report_v10.md';p.write_text(p.read_text()+'\n![Development comparison](artifacts/v10/validation_comparison_v10.png)\n')
print('Independent replay, analysis report, masking comparison and variable dictionaries saved.',flush=True)


Independent replay, analysis report, masking comparison and variable dictionaries saved.


Matplotlib is building the font cache; this may take a moment.


## 별도 원본 지표 우선 실험용 모델
주 모델의 스트레스 가드를 유지하고, 기존 v9 진단 규칙을 적용한 추가 실험용 분기입니다. 두 seed의 원본 F1 개선과 반복 관측한 홀드아웃은 독립 새 환자 성능이 아닙니다.

In [14]:
# Same clean-priority diagnostic rule documented in v9; primary adoption is unchanged.
experimental10=None
if not accepted10:
 eligible_experiment10=[]
 for name in finalists10:
  delta=deltas10[name];per=delta['per_seed'];mean=delta['mean']
  if all(values['clean']>=-.001 for values in per.values()) and mean['clean']>=.0015 and mean['noise']>=-.015:eligible_experiment10.append(name)
 if eligible_experiment10:
  exp_name10=max(eligible_experiment10,key=lambda name:deltas10[name]['mean']['clean']);exp_plan10=PLANS10[exp_name10]
  hcached=np.load(OUT/'holdout_probabilities.npz',allow_pickle=True);assert np.array_equal(hcached['indices'],hold)
  # The primary is v6, so this is a dev-only trained baseline, never a full-train model.
  ref_hold10=hcached['probabilities'];h_exp10=np.zeros_like(ref_hold10);hcache10={}
  for name,w in exp_plan10:
   if name=='v6_reference':h_exp10+=ref_hold10*w
   else:
    m=train10(name,dev,hcache10);h_exp10+=m.predict_frame(X.iloc[hold])*w
  h_pred10=labels[h_exp10.argmax(axis=1)]
  h_metrics_exp10=dict(macro_f1=f1_score(y[hold],h_pred10,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],h_pred10),weighted_f1=f1_score(y[hold],h_pred10,average='weighted',zero_division=0),n=len(hold))
  (OUT/'experimental_holdout_metrics.json').write_text(json.dumps(h_metrics_exp10,indent=2))
  # Fit the new member on all train only after the candidate and weights were frozen.
  with open(OUT/'cancer_model_v10.pkl','rb') as f:ref_bundle10=pickle.load(f)
  exp_models10=[];exp_weights10=[];cache_full_exp10={}
  for name,w in exp_plan10:
   if name=='v6_reference':exp_models10.extend(ref_bundle10['models']);exp_weights10.extend(ref_bundle10['weights']*w)
   else:exp_models10.append(train10(name,np.arange(len(train)),cache_full_exp10));exp_weights10.append(w)
  exp_weights10=np.asarray(exp_weights10,float)
  exp_bundle10=dict(version=10,experimental=True,models=exp_models10,weights=exp_weights10,genes=genes,classes=labels,winner=exp_name10,plan=exp_plan10,holdout_metrics=h_metrics_exp10,predict=predict_bundle10)
  exp_model_path10=OUT/'cancer_model_v10_clean_experimental.pkl'
  with exp_model_path10.open('wb') as f:cloudpickle.dump(exp_bundle10,f,protocol=pickle.HIGHEST_PROTOCOL)
  exp_prob10=predict_bundle10(exp_bundle10,test);assert np.isfinite(exp_prob10).all() and np.allclose(exp_prob10.sum(axis=1),1)
  exp_csv10=sample.copy();exp_csv10.SUBCLASS=labels[exp_prob10.argmax(axis=1)];exp_csvpath10=SUBMIT/'혜진_submission_v10_clean_experimental.csv';exp_csv10.to_csv(exp_csvpath10,index=False);exp_csv10.to_csv(OUT/'submission_v10_clean_experimental.csv',index=False)
  np.savez_compressed(OUT/'experimental_holdout_probabilities.npz',probabilities=h_exp10,indices=hold)
  pd.DataFrame(exp_prob10,index=test.ID,columns=labels).to_csv(OUT/'experimental_test_probabilities.csv')
  replay_script10="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);s=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert s.ID.equals(x.ID) and np.array_equal(b['classes'][p.argmax(axis=1)],s.SUBCLASS.to_numpy())
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
  env=os.environ.copy();env.pop('PYTHONPATH',None)
  with tempfile.TemporaryDirectory() as td:
   pf=Path(td)/'p.npy';r=subprocess.run([sys.executable,'-c',replay_script10,str(exp_model_path10.resolve()),str((DATA/'test.csv').resolve()),str(exp_csvpath10.resolve()),str(pf)],cwd=td,env=env,capture_output=True,text=True)
   if r.returncode:raise RuntimeError(r.stderr)
   assert np.allclose(np.load(pf),exp_prob10)
  reference_csv10=pd.read_csv(OUT/'submission_v10.csv');changed10=int((reference_csv10.SUBCLASS!=exp_csv10.SUBCLASS).sum())
  experimental10=dict(status='passed',experimental=True,candidate=exp_name10,plan=exp_plan10,primary_adopted=False,primary_cv=rank10.loc[exp_name10].to_dict(),baseline_cv=rank10.loc['v6_reference'].to_dict(),repeated_deltas=deltas10[exp_name10],holdout=h_metrics_exp10,baseline_holdout=metrics10,public_score=None,changed_predictions_vs_v6=changed10,model_bytes=exp_model_path10.stat().st_size,independent_reload='passed',csv_exact_match=True,batch_invariance=True,diagnostic_rule='v9 clean diagnostic: each-seed clean delta >= -0.001; mean clean >= +0.0015; mean noise >= -0.015')
  (OUT/'experimental_verification.json').write_text(json.dumps(experimental10,indent=2))
  wtable=pd.DataFrame([dict(model=m.name,weight=float(w),inputs=len(m.names)) for m,w in zip(exp_models10,exp_weights10)]);wtable.to_csv(OUT/'experimental_model_sizes.csv',index=False)
  new_features=[]
  for m,w in zip(exp_models10,exp_weights10):
   if m.name in C10:
    if hasattr(m.est,'feature_log_prob_'):imp=np.max(np.abs(m.est.feature_log_prob_),axis=0);metric='max_absolute_complement_log_weight'
    elif hasattr(m.est,'coef_'):imp=np.max(np.abs(m.est.coef_),axis=0);metric='max_absolute_coefficient'
    else:imp=m.est.feature_importances_;metric='tree_importance'
    for feature,value in zip(m.raw_names,imp):new_features.append(dict(model=m.name,feature=feature,importance=float(value),metric=metric,meaning='유전자별 변이 관측 존재' if feature.startswith('gene:') else '유전자별 변이 유형 관측 존재'))
  new_features=pd.DataFrame(new_features);new_features.to_csv(OUT/'experimental_new_features.csv',index=False)
  before=pd.DataFrame(classification_report(y[hold],labels[ref_hold10.argmax(axis=1)],output_dict=True,zero_division=0)).T.loc[labels]
  after=pd.DataFrame(classification_report(y[hold],h_pred10,output_dict=True,zero_division=0)).T.loc[labels]
  changes=pd.DataFrame(dict(class_code=labels,v6_f1=before['f1-score'].values,experimental_f1=after['f1-score'].values,support=before.support.values));changes['delta_f1']=changes.experimental_f1-changes.v6_f1;changes.to_csv(OUT/'experimental_holdout_class_comparison.csv',index=False)
  extra='\n## 원본 지표 우선 진단을 통과한 실험용 후보\n\n'
  extra+='기존 주 모델 규칙은 유지해 v6를 교체하지 않았습니다. 다만 v9에서 이미 정한 별도 진단 규칙(각 seed 원본 F1 하락 0.001 이하, 두 seed 평균 원본 +0.0015 이상, 평균 합성 스트레스 하락 0.015 이하)을 충족한 후보를 실험용으로 저장했습니다. 이 분기는 v10 반복 결과를 본 뒤 적용한 탐색이므로 선택 편향이 있고, 원래 주 모델 채택과 구분합니다.\n\n'
  extra+=f'실험용 `{exp_name10}`: `{exp_plan10}`. 가리기 증강이나 SVM이 최종 실험용에 모두 들어간 것은 아닙니다. 실제 선택된 것은 유전자+유형 이진 표현의 Complement NB(alpha=0.1)이며 기존 v6와 혼합했습니다.\n\n'
  extra+=mdtable10(confirmation10.reset_index())+'\n\n'
  extra+=f'후보/가중치를 고정하고 새 구성은 개발 환자로 학습한 뒤 기존 홀드아웃에서 평가했습니다. 유지 v6 F1 **{metrics10["macro_f1"]:.6f}**, 정확도 **{metrics10["accuracy"]:.6f}**; 실험용 F1 **{h_metrics_exp10["macro_f1"]:.6f}**, 정확도 **{h_metrics_exp10["accuracy"]:.6f}**입니다. 이 홀드아웃은 반복 관측한 값이고 결과로 비율을 다시 조정하지 않았습니다. 새 독립 평가나 Public 향상을 보장하지 않습니다.\n\n'
  extra+='실험용 모델별 가중치와 실제 입력 수:\n\n'+mdtable10(wtable)+'\n\n'
  extra+=f'제출 파일 submission/혜진_submission_v10_clean_experimental.csv, 모델 artifacts/v10/cancer_model_v10_clean_experimental.pkl입니다. v6 대비 예측 변경 {changed10}개는 추가 정답 수가 아닙니다. 독립 폴더에서 전체 test 2,546개 CSV·ID·확률·배치 일치를 확인했습니다. 자동 제출하지 않았으며 Public은 미확인입니다.\n\n'
  extra+='기존 홀드아웃 클래스별 변화(선택 후 참고 분석):\n\n'+mdtable10(changes.sort_values('delta_f1',ascending=False))+'\n'
  path=WORK/'analysis_report_v10.md';body=path.read_text();marker='\n## 원본 지표 우선 진단을 통과한 실험용 후보\n';path.write_text(body.split(marker)[0]+extra)
  path=WORK/'selected_variables_v10.md';body=path.read_text();marker='\n## 실험용 Complement NB의 추가 변수\n';path.write_text(body.split(marker)[0]+marker+'\n주 모델 유지 변수와 구분한 실험용 추가 입력입니다. 중요도는 다른 모델과 크기를 비교하거나 인과효과로 해석하지 않습니다. 기존 v6 입력은 위 목록과 같습니다.\n\n'+mdtable10(new_features)+'\n')
  print('V10 CLEAN EXPERIMENT SAVED:',exp_csvpath10,'holdout:',h_metrics_exp10,'changed:',changed10,flush=True)
 else:print('No candidate passed the existing clean diagnostic rule; no experimental CSV.')


V10 CLEAN EXPERIMENT SAVED: /Users/mac/project/cancer_project/submission/혜진_submission_v10_clean_experimental.csv holdout: {'macro_f1': 0.4738622192876446, 'accuracy': 0.4649476228847704, 'weighted_f1': 0.46788563815378725, 'n': 1241} changed: 92
